# 문제 2-1 : 콤마 구분 리스트 파서 활용 - 한국 추천 리스트

사용자가 관심 분야(예: 음식, 스포츠, 영화)를 입력하면 그 분야와 관련된 한국의 유명한 장소나 활동 5가지를 리스트로 출력합니다.

- `CommaSeparatedListOutputParser` 사용
- 사용자 입력을 받아 동적으로 처리
- 한국 관련 내용으로 추천

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("OpenAI Key:", "OK" if OPENAI_API_KEY else "없음", "| Groq Key:", "OK" if GROQ_API_KEY else "없음")

### LLM 제공자 선택

`PROVIDER` 값만 `"groq"` 또는 `"openai"`로 바꾸면 이후 모든 셀의 모델이 바뀝니다.
Groq는 OpenAI 호환 API라서 둘 다 `ChatOpenAI` 클래스를 사용하고 `base_url`만 다릅니다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

llm = get_llm(temperature=0.7)
output_parser = CommaSeparatedListOutputParser()

## 2. 형식 지침 확인

파서는 두 번 일합니다.
- **호출 전**: `get_format_instructions()`가 "콤마로 구분해서 답하라"는 지침 문자열을 만들고, 이 문자열을 프롬프트에 넣습니다.
- **호출 후**: 모델의 답변 문자열을 콤마 기준으로 나눠 `list[str]`로 바꿉니다.

In [ ]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

## 3. 프롬프트 + LCEL 체인

`partial_variables`로 형식 지침을 미리 채워 두면, 실행할 때는 `category`만 넘기면 됩니다.

In [ ]:
prompt = PromptTemplate(
    template="""당신은 한국 여행·문화 전문가입니다.
'{category}' 분야와 관련된 한국의 유명한 장소나 활동 5가지를 추천하세요.
각 항목은 '지역 + 이름' 형태로 짧게 쓰세요. (예: 전주 비빔밥)
이름에 지역이 이미 들어 있으면 지역을 반복하지 마세요. (예: 부산국제영화제)

{format_instructions}""",
    input_variables=["category"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser

## 4. 실행

In [ ]:
result = chain.invoke({"category": "음식"})
print(type(result))
print(result)

In [ ]:
categories = ["스포츠", "영화", "역사"]
results = chain.batch([{"category": c} for c in categories])

for category, items in zip(categories, results):
    print(f"[{category}] {items}")

## 5. 사용자 입력으로 실행

빈 값을 입력하면 종료합니다.

In [ ]:
while True:
    category = input("관심 분야를 입력하세요 (종료: Enter): ").strip()
    if not category:
        break
    print(f"[{category}] {chain.invoke({'category': category})}")